# Financial Retrieval Refinement: Threshold Filtering and MMR

This experiment evaluates whether similarity filtering or maximal marginal relevance (MMR) improves evidence selection over dense retrieval on FinDER. It also measures retrieval latency across sparse, dense, hybrid, and refined methods.

The benchmark contains 5,703 questions and 5,830 deduplicated reference passages, with 1,128 development questions and 4,575 held-out test questions. Reference passages are indexed without additional chunking.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
assert (ROOT / 'finder_hybrid_experiment.py').exists(), 'Open this notebook from the team repository root'
OUT = ROOT / 'results/d_refinement'

LABELS = {'d_dense_control': 'Dense', 'd_threshold_0.3': 'Dense + threshold (0.3)', 'd_mmr_1.0_10': 'MMR (lambda=1, fetch=10)'}


## Experimental setup

Passages and questions are encoded using `all-MiniLM-L6-v2`. Normalized vectors are ranked by cosine similarity, with a candidate pool of 100 passages. Retrieval is evaluated at k = 1, 3, 5, and 10; k = 5 is the primary comparison.

Relevance is defined by exact membership in each question's annotated references. Long passages are subject to the model's 256-token input limit.

In [2]:
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    from d_threshold_mmr import run
    run()
manifest = json.loads((OUT / 'run_manifest.json').read_text())
display(pd.DataFrame({'Setting': ['Embedding model', 'Reference passages', 'Development queries', 'Test queries', 'Candidate depth', 'Device'], 'Value': [manifest['model'], manifest['corpus'], manifest['dev'], manifest['test'], manifest['candidate_depth'], manifest['device']]}))


,Setting,Value
0,Embedding model,sentence-transformers/all-MiniLM-L6-v2
1,Reference passages,5830
2,Development queries,1128
3,Test queries,4575
4,Candidate depth,100
5,Device,cpu


## Parameter selection

The development sweep covers thresholds of 0.3, 0.5, and 0.7, and MMR weights of 0.3, 0.5, 0.7, and 1.0 with candidate counts of 10, 20, 50, and 100. Configurations are selected by Recall@5, with nDCG@5 as the tie-breaker.

The selected threshold is 0.3. The selected MMR configuration uses λ = 1.0 and 10 candidates. At this weight the diversity penalty is zero, so MMR reproduces dense ranking. The strongest configuration with an active diversity penalty, λ = 0.7 with 10 candidates, reaches development Recall@5 of 19.95%, below the dense baseline's 20.46%.

In [3]:
tuning = pd.read_csv(OUT / 'tuning_dev.csv')
display(tuning[['family', 'threshold', 'lambda_mult', 'fetch_k', 'recall_at_k', 'ndcg_at_k', 'mean_returned', 'empty_rate']].sort_values(['recall_at_k', 'ndcg_at_k'], ascending=False).round(5))


,family,threshold,lambda_mult,fetch_k,recall_at_k,ndcg_at_k,mean_returned,empty_rate
0,dense,NaN,NaN,NaN,0.20464,0.16210,5.00000,0.00000
1,threshold,0.3,NaN,NaN,0.20464,0.16210,4.97784,0.00089
16,mmr,NaN,1.0,10.0,0.20464,0.16210,5.00000,0.00000
17,mmr,NaN,1.0,20.0,0.20464,0.16210,5.00000,0.00000
18,mmr,NaN,1.0,50.0,0.20464,0.16210,5.00000,0.00000
19,mmr,NaN,1.0,100.0,0.20464,0.16210,5.00000,0.00000
12,mmr,NaN,0.7,10.0,0.19947,0.15763,5.00000,0.00000
13,mmr,NaN,0.7,20.0,0.18794,0.15229,5.00000,0.00000
14,mmr,NaN,0.7,50.0,0.18307,0.15026,5.00000,0.00000
15,mmr,NaN,0.7,100.0,0.17952,0.14853,5.00000,0.00000


## Test-set retrieval performance

Dense retrieval achieves Recall@5 of 21.1767% and nDCG@5 of 0.16889. Threshold filtering reduces the average returned count from 5.000 to 4.972, with Recall@5 falling slightly to 21.1694%. The selected MMR configuration matches dense retrieval.

Precision@5 uses a fixed denominator of five. Precision among returned passages measures the fraction of retained evidence that is relevant; empty results receive zero.

In [4]:
quality = pd.read_csv(OUT / 'test_metrics.csv')
primary = quality[quality.k == 5].copy()
primary['method'] = primary['method'].replace(LABELS)
display(primary[['method', 'precision_at_k', 'recall_at_k', 'mrr_at_k', 'ndcg_at_k', 'mean_returned']].rename(columns={'method': 'Method', 'precision_at_k': 'Precision@5', 'recall_at_k': 'Recall@5', 'mrr_at_k': 'MRR@5', 'ndcg_at_k': 'nDCG@5', 'mean_returned': 'Mean passages'}).round(5))


,Method,Precision@5,Recall@5,MRR@5,nDCG@5,Mean passages
2,Dense,0.04507,0.21177,0.15833,0.16889,5.00000
6,Dense + threshold (0.3),0.04503,0.21169,0.15833,0.16885,4.97246
10,"MMR (lambda=1, fetch=10)",0.04507,0.21177,0.15833,0.16889,5.00000


## Retrieval latency

Latency is measured on the same 100 test questions, using three warm-up calls per method and three randomly interleaved repetitions. Each measurement includes question encoding where applicable, search, fusion or refinement, and evidence lookup. Initialization and answer generation are excluded.

On this CPU, mean single-query latency is 17.17 ms for BM25, 19.93 ms for dense retrieval, and 37.29 ms for hybrid retrieval. Threshold and MMR take approximately 20 ms. The small differences among dense variants do not establish a reliable speed advantage.

Batch-average latency is retained separately in the exported metrics and is not interchangeable with single-query latency.

In [5]:
latency = pd.read_csv(OUT / 'latency_single_query.csv')
latency['method'] = latency['method'].replace(LABELS)
display(latency.rename(columns={'method': 'Method', 'mean_ms': 'Mean (ms)', 'median_ms': 'Median (ms)', 'p95_ms': 'p95 (ms)', 'measurements': 'Measurements'}).round(2))


,Method,Mean (ms),Median (ms),p95 (ms),Measurements
0,BM25,17.17,16.90,19.42,300
1,Hybrid,37.29,36.80,43.31,300
2,Dense,19.93,19.49,24.60,300
3,"MMR (lambda=1, fetch=10)",20.05,19.75,24.46,300
4,Dense + threshold (0.3),20.32,19.95,24.51,300


## Evidence changes

At k = 5, threshold filtering removes gold evidence for one test question and leaves gold coverage unchanged for the other 4,574. Six questions return no passages. Selected MMR leaves gold coverage unchanged for all 4,575 questions.

The records below identify evidence changes relative to dense retrieval. These labels describe changes in gold coverage; they are not a manual classification of failure causes.

In [6]:
errors = pd.read_csv(OUT / 'paired_errors.csv')
counts = errors.groupby(['method', 'status']).size().rename('Queries').reset_index()
counts['method'] = counts['method'].replace(LABELS)
display(counts.rename(columns={'method': 'Method', 'status': 'Gold coverage'}))
display(errors[errors.status != 'unchanged'][['question', 'category', 'gold_ids', 'baseline_ids', 'refined_ids', 'lost_gold']])


,Method,Gold coverage,Queries
0,"MMR (lambda=1, fetch=10)",unchanged,4575
1,Dense + threshold (0.3),unchanged,4574
2,Dense + threshold (0.3),worsened,1


,question,category,gold_ids,baseline_ids,refined_ids,lost_gold
185,MTCH's (Match Group) rev mix impacts its val & growth prospects.,Footnotes,"[260, 262, 263]","[262, 145, 3678, 4726, 260]",[262],[260]


In [7]:
def inspect_case(query_index, method=None):
    from finder_hybrid_experiment import build_reference_corpus
    data = pd.read_parquet(ROOT / 'data/train-00000-of-00001.parquet')
    corpus, _ = build_reference_corpus(data)
    selected = errors[errors.query_index == query_index]
    if method is not None:
        selected = selected[selected.method == method]
    for row in selected.itertuples():
        print(row.method, row.question)
        for label in ['gold_ids','baseline_ids','refined_ids']:
            print('\n' + label)
            for pid in json.loads(getattr(row,label)):
                print(f'[{pid}] {corpus[pid]}\n')



## Findings

Neither refinement improves held-out retrieval quality in this experiment. A low threshold removes little context, while higher thresholds reduce development-set recall substantially. Diversity-active MMR configurations also underperform the dense baseline on development questions.

These findings apply to pooled FinDER reference passages with MiniLM embeddings. They do not establish performance on full filings or the effect on generated answers.